### 1. Download/Load SP500 stocks prices data.

In [6]:
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')


In [7]:
import pandas as pd
import requests
import certifi
from io import StringIO

# URL of the Wikipedia page containing the list of S&P 500 companies

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# make a GET request to the URL with a user-agent header and SSL verification using certifi

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, verify=certifi.where())

# check if the request was successful

html = response.text

# read all tables from the HTML content

tables = pd.read_html(StringIO(html))

# name first table as sp500

sp500 = tables[0]

# clean ticker symbols by replacing '.' with '-'

sp500['Symbol'] = sp500["Symbol"].str.replace('.', '-')

# get unique ticker symbols as a list

symbols_list = sp500["Symbol"].unique().tolist()

# set the end date for data retrieval and calculate the start date as 8 years prior

end_date = '2026-03-20'

# calculate the start date as 8 years prior to the end date using pd.DateOffset

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

df = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date,
    auto_adjust=False  # 👈 IMPORTANT
).stack()

df.index.names = ['date', 'ticker']

df.columns= df.columns.str.lower()

df

[*********************100%***********************]  503 of 503 completed


Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.400593   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667408   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621094   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924835   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.596695   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  
date       ticker               
2018-03-22 A         1690000.0  
           AAPL    165963200.0  
           ABBV     26861200.0  
           ABNB            NaN  
           ABT       5345300.0  
...                        ...  
2026-03-19 XYZ       6387300.0  
           YUM       1832100.0  
           ZBH       1812000.0  
           ZBRA       593300.0  
           ZTS       4999100.0  

[1010527 rows x 6 columns]

### 2. Calculate features and technical indicators for each stock.
- Garman-Klass Volatility
- RSI
- Bollinger Bands
- ATR
- MACD
- Dollar Volume

In [8]:
# Columns to ensure are numeric
cols_to_convert = ['open', 'high', 'low', 'close', 'adj close', 'volume']

# Convert specified columns to numeric, coercing errors to NaN
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

In [9]:
# Force all numeric columns to float first
numeric_cols = ['open', 'high', 'low', 'close', 'adj close', 'volume']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Garman-Klass Volatility
df['garman_klass_vol'] = (
    (np.log(df['high']) - np.log(df['low']))**2
) / 2 - (2 * np.log(2) - 1) * (
    np.log(df['adj close']) - np.log(df['open'])
)**2

# RSI
df['rsi'] = df.groupby(level=1)['adj close'].transform(
    lambda x: pandas_ta.rsi(close=x.astype(float), length=20)
)

# Bollinger Bands
def compute_bbands(x):
    bb = pandas_ta.bbands(close=np.log1p(x.astype(float)), length=20)
    if bb is None:
        return pd.DataFrame(np.nan, index=x.index, columns=[0, 1, 2])
    return bb

df['bb_low'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 0]
)
df['bb_mid'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 1]
)
df['bb_high'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 2]
)

# ATR
def compute_atr(stock_data):
    try:
        atr = pandas_ta.atr(
            high=stock_data['high'].astype(float),
            low=stock_data['low'].astype(float),
            close=stock_data['adj close'].astype(float),
            length=14
        )
        if atr is None or atr.isna().all():
            return pd.Series(np.nan, index=stock_data.index)
        atr = atr.sub(atr.mean()).div(atr.std())
        atr.index = stock_data.index
        return atr
    except Exception:
        return pd.Series(np.nan, index=stock_data.index)

df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# MACD
def compute_macd(close):
    try:
        macd = pandas_ta.macd(close=close.astype(float), length=20).iloc[:, 0]
        return macd.sub(macd.mean()).div(macd.std())
    except Exception:
        return pd.Series(np.nan, index=close.index)

df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

# Dollar Volume
df['dollar_volume'] = (df['adj close'] * df['volume']) / 1e6

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.400593   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667408   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621094   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924835   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.596695   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2018-03-22 A         1690000.0         -0.002114        NaN       NaN   
           AAPL    165963200.0         -0.001552        NaN       NaN   
           ABBV     26861200.0         -0.058747        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT       5345300.0         -0.009116        NaN       NaN   
...                        ...               ...        ...       ...   
2026-03-19 XYZ       6387300.0          0.000407  47.622919  3.927554   
           YUM       1832100.0          0.000139  45.700865  5.048697   
           ZBH       1812000.0          0.000238  42.287766  4.488886   
           ZBRA       593300.0          0.000443  37.090520  5.274545   
           ZTS       4999100.0          0.000207  39.486943  4.737676   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2018-03-22 A            NaN       NaN       NaN       NaN     107.147002  
           AAPL         NaN       NaN       NaN       NaN    6583.329966  
           ABBV         NaN       NaN       NaN       NaN    1870.106123  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     282.899122  
...                     ...       ...       ...       ...            ...  
2026-03-19 XYZ     4.109759  4.291963 -0.541365  0.056086     376.786838  
           YUM     5.090827  5.132957 -1.732438 -0.236914     286.265625  
           ZBH     4.564811  4.640737 -1.397500 -0.345068     162.349211  
           ZBRA    5.401648  5.528752  0.095328 -1.256195     122.332528  
           ZTS     4.821242  4.904809 -1.780040 -1.013602     579.845598  

[1010527 rows x 14 columns]

### 3. Aggregate to monthly level and filter top 150 most liquid stocks for each month
- To reduce training time and experiment with features and strategies, we convert the business-daily data to month-end frequency

In [10]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]

data = pd.concat([
    df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
    df.unstack()[last_cols].resample('ME').last().stack('ticker')
], axis=1).dropna(subset=['dollar_volume', 'adj close'])  # only drop if these are NaN

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-03-31 A          142.949879   62.864994         -0.001147   8.661233   
           AAPL      6347.315194   39.416027         -0.001092  10.343328   
           ABBV       956.063235   67.172653         -0.044830  14.218689   
           ABT        332.180028   52.047527         -0.006877   7.529451   
           ACGL        64.065371   27.129135         -0.001026  12.204337   
...                          ...         ...               ...        ...   
2026-03-31 XYZ        581.353228   58.990002          0.000407  47.622919   
           YUM        279.860158  156.250000          0.000139  45.700865   
           ZBH        199.530890   89.596695          0.000238  42.287766   
           ZBRA       162.709509  206.190002          0.000443  37.090520   
           ZTS        481.208041  115.989998          0.000207  39.486943   

                     bb_low    bb_mid   bb_high       atr      macd  
date       ticker                                                    
2018-03-31 A            NaN       NaN       NaN       NaN       NaN  
           AAPL         NaN       NaN       NaN       NaN       NaN  
           ABBV         NaN       NaN       NaN       NaN       NaN  
           ABT          NaN       NaN       NaN       NaN       NaN  
           ACGL         NaN       NaN       NaN       NaN       NaN  
...                     ...       ...       ...       ...       ...  
2026-03-31 XYZ     3.927554  4.109759  4.291963 -0.541365  0.056086  
           YUM     5.048697  5.090827  5.132957 -1.732438 -0.236914  
           ZBH     4.488886  4.564811  4.640737 -1.397500 -0.345068  
           ZBRA    5.274545  5.401648  5.528752  0.095328 -1.256195  
           ZTS     4.737676  4.821242  4.904809 -1.780040 -1.013602  

[47837 rows x 9 columns]

- Calculate 5-year rolling average of dollar volume for each stocks before filtering.

In [11]:
data['dollar_volume'] = (data.loc[:,'dollar_volume'].unstack('ticker').rolling(5*12, min_periods=12).mean().stack('ticker')
    .reindex(data.index))

data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2019-02-28 AAPL     41.296986         -0.001053  57.086259  3.705447   
           ABBV     58.724594         -0.035250  42.829346  4.080877   
           ABT      68.647751         -0.005012  64.682193  4.161157   
           ACN     145.995926         -0.003600  63.333925  4.938362   
           ADBE    262.500000          0.000144  62.689552  5.521170   
...                       ...               ...        ...       ...   
2026-03-31 WDC     316.929993          0.002351  62.481774  5.485216   
           WFC      76.389999          0.000358  35.566775  4.293305   
           WMT     120.841995          0.000350  45.694634  4.804058   
           XOM     158.160004          0.000451  65.453187  4.986162   
           XYZ      58.990002          0.000407  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2019-02-28 AAPL    3.733820  3.762194 -1.654845 -0.000532  
           ABBV    4.097572  4.114267  0.821915 -0.614993  
           ABT     4.204768  4.248378  0.532867  0.884587  
           ACN     4.970015  5.001669  0.092059  0.415526  
           ADBE    5.554233  5.587295 -1.502455  0.466103  
...                     ...       ...       ...       ...  
2026-03-31 WDC     5.622986  5.760756  5.844734  3.115020  
           WFC     4.400714  4.508124 -0.948356 -3.360090  
           WMT     4.835141  4.866224  0.219226 -0.703873  
           XOM     5.032413  5.078663 -1.321983  2.399187  
           XYZ     4.109759  4.291963 -0.541365  0.056086  

[12814 rows x 8 columns]

### 4. Calculate Monthly Returns for different time horizons as features.

- To capture time series dynamics that reflect, for example, momentum patterns, we compute historical returns using the method .pct_change(lag). returns over various monthly periods as identified by lags.

In [12]:
def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1,2,3,6,9,12]

    for lag in lags:
        df[f'return_{lag}m'] = (df['adj close']
                            .pct_change(lag)
                           .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                  upper=x.quantile(1-outlier_cutoff)))
                            .add(1)
                            .pow(1/lag)
                            .sub(1))
    return df

data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data



adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2020-02-29 AAPL     66.050842          0.003127  34.635026  4.220314   
           ABBV     67.191643         -0.015615  43.432041  4.163786   
           ABT      69.228966         -0.003020  26.562262  4.276588   
           ACN     166.075134         -0.001623  24.475319  5.156322   
           ADBE    345.119995          0.000579  45.805778  5.830231   
...                       ...               ...        ...       ...   
2026-03-31 WDC     316.929993          0.002351  62.481774  5.485216   
           WFC      76.389999          0.000358  35.566775  4.293305   
           WMT     120.841995          0.000350  45.694634  4.804058   
           XOM     158.160004          0.000451  65.453187  4.986162   
           XYZ      58.990002          0.000407  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2020-02-29 AAPL    4.331205  4.442096 -0.674765 -0.865349  -0.114702   
           ABBV    4.274760  4.385733  0.623538 -0.176212   0.057887   
           ABT     4.366030  4.455472  0.532680 -1.484042  -0.116020   
           ACN     5.254162  5.352003  0.387223 -0.878929  -0.119975   
           ADBE    5.901527  5.972823 -0.284170  0.001727  -0.017144   
...                     ...       ...       ...       ...        ...   
2026-03-31 WDC     5.622986  5.760756  5.844734  3.115020   0.133649   
           WFC     4.400714  4.508124 -0.948356 -3.360090  -0.062124   
           WMT     4.835141  4.866224  0.219226 -0.703873  -0.053615   
           XOM     5.032413  5.078663 -1.321983  2.399187   0.037115   
           XYZ     4.109759  4.291963 -0.541365  0.056086  -0.073940   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2020-02-29 AAPL    -0.034022   0.008360   0.046912   0.051828    0.039912  
           ABBV    -0.009487  -0.003286   0.050073   0.017255    0.011287  
           ABT     -0.056286  -0.032622  -0.015540   0.002640    0.000703  
           ACN     -0.072132  -0.034077  -0.014026   0.002469    0.010796  
           ADBE     0.022947   0.036945   0.032711   0.027270    0.023065  
...                      ...        ...        ...        ...         ...  
2026-03-31 WDC      1.801266   0.935565   0.366872   0.209807    0.163347  
           WFC     -0.078951  -0.062618  -0.013684  -0.003513    0.006989  
           WMT      0.008150   0.028162   0.027583   0.024539    0.027742  
           XOM      0.061124   0.097802   0.060737   0.046370    0.026927  
           XYZ     -0.011986  -0.032269  -0.033274  -0.015557    0.006881  

[10574 rows x 14 columns]

### 5. Download Fama-French Factors and Calculate Rolling Factor Betas.

- We will introduce the Fama-French data to estimate the exposure of assets to common risk factors using linear regressionn.
- The five Fama-French factors, namely market risk, size, value, operating profitability, and investment have been shown empirically to explain asset returns and are commonly used to assess the risk/return profile of portfolios. Hence, it is natural to include past factor exposures as financial features in models.
- We can access the historical factor returns using the pandas-datareader and estimate historical exposures using the RollingOLS rolling linear regression.

In [13]:
import zipfile, io, requests

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip"

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

ff_factors = pd.read_csv(
    z.open(z.namelist()[0]),
    skiprows=3,
    index_col=0
)

# The file has annual data appended at the bottom, cut it off
ff_factors = ff_factors[ff_factors.index.astype(str).str.strip().str.match(r'^\d{6}$')]

# Convert index to datetime to match your data's date index
ff_factors.index = pd.to_datetime(ff_factors.index, format='%Y%m') + pd.offsets.MonthEnd(0)

# Filter from 2010 onwards
ff_factors = ff_factors.loc['2010':]

# drop RF column and divide by 100
factor_data = ff_factors.apply(pd.to_numeric, errors='coerce').drop('RF', axis=1).div(100)

factor_data.index.name = 'date'

factor_data = factor_data.join(data['return_1m']).sort_index()

factor_data




Mkt-RF     SMB     HML     RMW     CMA  return_1m
date       ticker                                                   
2020-02-29 AAPL   -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.114702
           ABBV   -0.0815  0.0008 -0.0382 -0.0143 -0.0253   0.057887
           ABT    -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.116020
           ACN    -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.119975
           ADBE   -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.017144
...                   ...     ...     ...     ...     ...        ...
2026-02-28 WDC    -0.0117  0.0063  0.0283  0.0162  0.0507   5.149144
           WFC    -0.0117  0.0063  0.0283  0.0162  0.0507  -0.095477
           WMT    -0.0117  0.0063  0.0283  0.0162  0.0507   0.073947
           XOM    -0.0117  0.0063  0.0283  0.0162  0.0507   0.085689
           XYZ    -0.0117  0.0063  0.0283  0.0162  0.0507   0.054112

[10430 rows x 6 columns]

- Filter out stocks with less than 10 months of data.

In [14]:
observations = factor_data.groupby(level=1).size()
valid_stocks = observations[observations >= 10]
factor_data = factor_data[factor_data.index.get_level_values("ticker").isin(valid_stocks.index)]
factor_data

Mkt-RF     SMB     HML     RMW     CMA  return_1m
date       ticker                                                   
2020-02-29 AAPL   -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.114702
           ABBV   -0.0815  0.0008 -0.0382 -0.0143 -0.0253   0.057887
           ABT    -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.116020
           ACN    -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.119975
           ADBE   -0.0815  0.0008 -0.0382 -0.0143 -0.0253  -0.017144
...                   ...     ...     ...     ...     ...        ...
2026-02-28 WDC    -0.0117  0.0063  0.0283  0.0162  0.0507   5.149144
           WFC    -0.0117  0.0063  0.0283  0.0162  0.0507  -0.095477
           WMT    -0.0117  0.0063  0.0283  0.0162  0.0507   0.073947
           XOM    -0.0117  0.0063  0.0283  0.0162  0.0507   0.085689
           XYZ    -0.0117  0.0063  0.0283  0.0162  0.0507   0.054112

[10383 rows x 6 columns]

- calculate rolling factor betas

In [15]:
betas = (factor_data.groupby(level=1,
                    group_keys=False).apply(lambda x: RollingOLS(endog=x['return_1m'], exog=sm.add_constant(x.drop('return_1m', axis=1)), window=min(24, x.shape[0]), min_nobs=len(x.columns)+1).fit(params_only=True).params.drop('const', axis=1)))

betas

Mkt-RF       SMB       HML       RMW        CMA
date       ticker                                                   
2020-02-29 AAPL         NaN       NaN       NaN       NaN        NaN
           ABBV         NaN       NaN       NaN       NaN        NaN
           ABT          NaN       NaN       NaN       NaN        NaN
           ACN          NaN       NaN       NaN       NaN        NaN
           ADBE         NaN       NaN       NaN       NaN        NaN
...                     ...       ...       ...       ...        ...
2026-02-28 WDC    -0.573704  8.052432 -4.787903  7.856691  17.892516
           WFC     1.086136 -0.335074  0.885972 -1.388747  -0.434126
           WMT     1.012595 -0.095837  0.170044  0.470634  -0.347065
           XOM     0.390731 -0.092940  1.211706  0.166327   0.508860
           XYZ     0.970831  1.158832 -1.629897 -2.096030   1.186771

[10383 rows x 5 columns]

In [16]:
betas.groupby('ticker').shift()

Mkt-RF       SMB       HML       RMW       CMA
date       ticker                                                  
2020-02-29 AAPL         NaN       NaN       NaN       NaN       NaN
           ABBV         NaN       NaN       NaN       NaN       NaN
           ABT          NaN       NaN       NaN       NaN       NaN
           ACN          NaN       NaN       NaN       NaN       NaN
           ADBE         NaN       NaN       NaN       NaN       NaN
...                     ...       ...       ...       ...       ...
2026-02-28 WDC          NaN       NaN       NaN       NaN       NaN
           WFC     1.037456 -0.240968  0.670651 -1.475123  0.046386
           WMT     1.118517 -0.199146  0.351640  0.536377 -0.888180
           XOM     0.476254 -0.122051  1.204760  0.154002  0.341947
           XYZ     1.222330  1.033301 -1.537417 -2.084186  0.497421

[10383 rows x 5 columns]

In [17]:
factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

data[factors] = betas.groupby('ticker').shift()[factors]

data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

data = data.dropna()

data.info()

<class 'pandas.DataFrame'>
MultiIndex: 10238 entries, (Timestamp('2020-02-29 00:00:00'), 'AAPL') to (Timestamp('2026-03-31 00:00:00'), 'XYZ')
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   adj close         10238 non-null  float64
 1   garman_klass_vol  10238 non-null  float64
 2   rsi               10238 non-null  float64
 3   bb_low            10238 non-null  float64
 4   bb_mid            10238 non-null  float64
 5   bb_high           10238 non-null  float64
 6   atr               10238 non-null  float64
 7   macd              10238 non-null  float64
 8   return_1m         10238 non-null  float64
 9   return_2m         10238 non-null  float64
 10  return_3m         10238 non-null  float64
 11  return_6m         10238 non-null  float64
 12  return_9m         10238 non-null  float64
 13  return_12m        10238 non-null  float64
 14  Mkt-RF            10238 non-null  float64
 15  SMB               102

In [18]:
from sklearn.cluster import KMeans

def get_clusters(df):
    df['cluster'] = KMeans(n_clusters=4, 
                           random_state=0, 
                           init='random').fit(df).labels_
    
    return df
    
data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2020-02-29 AAPL     66.050842          0.003127  34.635026  4.220314   
           ABBV     67.191643         -0.015615  43.432041  4.163786   
           ABT      69.228966         -0.003020  26.562262  4.276588   
           ACN     166.075134         -0.001623  24.475319  5.156322   
           ADBE    345.119995          0.000579  45.805778  5.830231   
...                       ...               ...        ...       ...   
2026-03-31 WDAY    133.380005          0.000661  35.589722  4.841546   
           WFC      76.389999          0.000358  35.566775  4.293305   
           WMT     120.841995          0.000350  45.694634  4.804058   
           XOM     158.160004          0.000451  65.453187  4.986162   
           XYZ      58.990002          0.000407  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2020-02-29 AAPL    4.331205  4.442096 -0.674765 -0.865349  -0.114702   
           ABBV    4.274760  4.385733  0.623538 -0.176212   0.057887   
           ABT     4.366030  4.455472  0.532680 -1.484042  -0.116020   
           ACN     5.254162  5.352003  0.387223 -0.878929  -0.119975   
           ADBE    5.901527  5.972823 -0.284170  0.001727  -0.017144   
...                     ...       ...       ...       ...        ...   
2026-03-31 WDAY    4.930753  5.019961  0.253458 -1.366942  -0.002841   
           WFC     4.400714  4.508124 -0.948356 -3.360090  -0.062124   
           WMT     4.835141  4.866224  0.219226 -0.703873  -0.053615   
           XOM     5.032413  5.078663 -1.321983  2.399187   0.037115   
           XYZ     4.109759  4.291963 -0.541365  0.056086  -0.073940   

                   return_2m  return_3m  return_6m  return_9m  return_12m  \
date       ticker                                                           
2020-02-29 AAPL    -0.034022   0.008360   0.046912   0.051828    0.039912   
           ABBV    -0.009487  -0.003286   0.050073   0.017255    0.011287   
           ABT     -0.056286  -0.032622  -0.015540   0.002640    0.000703   
           ACN     -0.072132  -0.034077  -0.014026   0.002469    0.010796   
           ADBE     0.022947   0.036945   0.032711   0.027270    0.023065   
...                      ...        ...        ...        ...         ...   
2026-03-31 WDAY    -0.128543  -0.146836  -0.093725  -0.063186   -0.045603   
           WFC     -0.078951  -0.062618  -0.013684  -0.003513    0.006989   
           WMT      0.008150   0.028162   0.027583   0.024539    0.027742   
           XOM      0.061124   0.097802   0.060737   0.046370    0.026927   
           XYZ     -0.011986  -0.032269  -0.033274  -0.015557    0.006881   

                     Mkt-RF       SMB       HML       RMW       CMA  cluster  
date       ticker                                                             
2020-02-29 AAPL    1.077460  0.046517 -0.523404  0.480787  0.113426        0  
           ABBV    0.531432 -0.181963  0.243269  0.142293  0.733141        0  
           ABT     0.788835 -0.154545 -0.062260 -0.102403  0.769661        0  
           ACN     1.112179 -0.105654 -0.202697  0.074262 -0.169773        1  
           ADBE    1.502961 -0.764852 -0.020498  0.285766 -0.365166        2  
...                     ...       ...       ...       ...       ...      ...  
2026-03-31 WDAY    1.119326 -0.747503 -0.180547 -1.140597 -0.300861        1  
           WFC     1.046172 -0.114918  1.395592 -0.949769 -0.902014        1  
           WMT     0.701110 -0.185747 -0.307253  0.296718  0.198153        1  
           XOM     0.788181 -0.267366  1.017306 -0.145097  0.322113        1  
           XYZ     2.368750  0.461887 -0.129464 -2.602367  0.063779        1  

[10238 rows x 20 columns]

In [19]:

import sys
print(sys.executable)

c:\Users\jgyou\Quant-Finance-Repository\quant-finance\Scripts\python.exe
